# 1

In [1]:
import pandas as pd
import numpy as np
import ast

from sklearn.cluster import KMeans
from scipy.spatial.distance import jensenshannon

# 2

In [2]:
def parse_embedding(x):

    if isinstance(x, str):
        emb = ast.literal_eval(x)
    else:
        emb = x

    emb = np.array(emb)

    if emb.ndim > 1:
        emb = emb.flatten()

    return emb

# 3

In [3]:
TRAIN_PATH = "/kaggle/input/datasets/xparamx/sample-111/Climate.csv"

train_df = pd.read_csv(TRAIN_PATH)

print("Training samples:", len(train_df))
train_df.head()

Training samples: 128477


,date,sentence_id,main_sentence,w3_embedding,War,Health,Technology,Climate,Economics
0,2011-09-21 13:59:11,f1_a50_s1,Story highlightsUrsula Sladek created Germany'...,"-0.020415567,0.01938476,0.006955214,0.05260677...",0.180876,0.145461,0.236585,0.470581,0.172328
1,2011-09-21 13:59:11,f1_a50_s2,"Founded by Sladek and a few friends in 1991, E...","-0.02180171,0.017887823,0.010610466,0.04336527...",0.180741,0.126097,0.217226,0.444158,0.151267
2,2011-09-21 13:59:11,f1_a50_s3,"Not bad for a self-professed housewife, who wr...","-0.012865681,0.0292067,0.012496701,0.037087686...",0.183976,0.133819,0.200109,0.348209,0.085087
3,2011-09-21 13:59:11,f1_a50_s7,"""And we still have this now, when you go into ...","0.0071375067,0.110946946,-0.01228442,0.0051183...",0.240931,0.219310,0.119873,0.342446,0.071015
4,2011-09-21 13:59:11,f1_a50_s8,"Sladek, a mother of five, established ""Parents...","0.019184,0.09541876,0.0026660592,-0.0005222882...",0.254690,0.212780,0.161057,0.441645,0.108469


# 4

In [4]:
train_df["embedding"] = train_df["w3_embedding"].apply(parse_embedding)

print("Example embedding length:", len(train_df["embedding"].iloc[0]))

Example embedding length: 768


# 5

In [5]:
TOPIC = "Climate"

topic_cols = ["War","Health","Technology","Climate","Economics"]

train_df["dominant_topic"] = train_df[topic_cols].idxmax(axis=1)

train_topic_df = train_df[train_df["dominant_topic"] == TOPIC].copy()

print("Training Topic:", TOPIC)
print("Rows after filtering:", len(train_topic_df))

Training Topic: Climate
Rows after filtering: 91685


# 6

In [6]:
X_train = np.vstack(train_topic_df["embedding"])

print("Training matrix shape:", X_train.shape)

Training matrix shape: (91685, 768)


# 7

In [7]:
NUM_CLUSTERS = 5

kmeans = KMeans(
    n_clusters=NUM_CLUSTERS,
    random_state=42,
    n_init=10
)

kmeans.fit(X_train)

print("KMeans training complete")

KMeans training complete


# NEW

In [8]:
import pandas as pd

# use same dataset you already loaded
test_df = train_df.sample(30, random_state=42)

# save test dataset
test_df.to_csv("/kaggle/working/testing_sample_with_embeddings.csv", index=False)

print("Testing dataset created")
test_df.head()

Testing dataset created


,date,sentence_id,main_sentence,w3_embedding,War,Health,Technology,Climate,Economics,embedding,dominant_topic
78689,2023-12-28 22:19:00,f70_a450_s2,It is only the second company in mainland Chin...,"0.03697746,-0.033790104,-0.043182135,-0.012994...",0.133655,0.147496,0.177032,0.484364,0.154352,"[0.03697746, -0.033790104, -0.043182135, -0.01...",Climate
30065,2019-05-08 00:41:43,f22_a658_s20,But critics say that's only because it's carry...,"-0.01430066,0.074648865,0.0048698676,-0.000750...",0.172625,0.114172,0.093944,0.416087,0.237503,"[-0.01430066, 0.074648865, 0.0048698676, -0.00...",Climate
99681,2024-07-05 15:19:00,f84_a182_s13,He stated that most of the incidents recorded ...,"0.0019156963,0.028779268,-0.019089464,0.002254...",0.388295,0.218112,0.136699,0.350551,0.180355,"[0.0019156963, 0.028779268, -0.019089464, 0.00...",War
74917,2023-11-03 06:37:34,f66_a106_s20,The critical discussions would remain on struc...,"-0.02439864,0.008451387,-0.02472313,0.02484308...",0.376960,0.139569,0.153860,0.300916,0.503767,"[-0.02439864, 0.008451387, -0.02472313, 0.0248...",Economics
70546,2023-11-02 11:39:18,f62_a741_s99,Topics related to the future of CXOs explore t...,"0.044725332,0.014079512,-0.04256062,-0.0520905...",0.183970,0.206179,0.507275,0.310897,0.284260,"[0.044725332, 0.014079512, -0.04256062, -0.052...",Technology


# 8

In [9]:
import pickle

pickle.dump(kmeans, open("kmeans_narrative_model.pkl","wb"))

print("Model saved")

Model saved


# 9

In [10]:
TEST_PATH = "/kaggle/input/datasets/xparamx/testing-file/testing_sample.csv"

test_df = pd.read_csv("/kaggle/working/testing_sample_with_embeddings.csv")

print("Test samples:", len(test_df))
test_df.head()

Test samples: 30


,date,sentence_id,main_sentence,w3_embedding,War,Health,Technology,Climate,Economics,embedding,dominant_topic
0,2023-12-28 22:19:00,f70_a450_s2,It is only the second company in mainland Chin...,"0.03697746,-0.033790104,-0.043182135,-0.012994...",0.133655,0.147496,0.177032,0.484364,0.154352,[ 3.69774600e-02 -3.37901040e-02 -4.31821350e-...,Climate
1,2019-05-08 00:41:43,f22_a658_s20,But critics say that's only because it's carry...,"-0.01430066,0.074648865,0.0048698676,-0.000750...",0.172625,0.114172,0.093944,0.416087,0.237503,[-1.43006600e-02 7.46488650e-02 4.86986760e-...,Climate
2,2024-07-05 15:19:00,f84_a182_s13,He stated that most of the incidents recorded ...,"0.0019156963,0.028779268,-0.019089464,0.002254...",0.388295,0.218112,0.136699,0.350551,0.180355,[ 1.91569630e-03 2.87792680e-02 -1.90894640e-...,War
3,2023-11-03 06:37:34,f66_a106_s20,The critical discussions would remain on struc...,"-0.02439864,0.008451387,-0.02472313,0.02484308...",0.376960,0.139569,0.153860,0.300916,0.503767,[-2.43986400e-02 8.45138700e-03 -2.47231300e-...,Economics
4,2023-11-02 11:39:18,f62_a741_s99,Topics related to the future of CXOs explore t...,"0.044725332,0.014079512,-0.04256062,-0.0520905...",0.183970,0.206179,0.507275,0.310897,0.284260,[ 4.47253320e-02 1.40795120e-02 -4.25606200e-...,Technology


# 10

In [11]:
test_df["embedding"] = test_df["w3_embedding"].apply(parse_embedding)

X_test = np.vstack(test_df["embedding"])

print("Test matrix shape:", X_test.shape)

Test matrix shape: (30, 768)


# 11

In [12]:
test_df["cluster"] = kmeans.predict(X_test)

test_df[["date", "main_sentence", "cluster"]].head()

,date,main_sentence,cluster
0,2023-12-28 22:19:00,It is only the second company in mainland Chin...,4
1,2019-05-08 00:41:43,But critics say that's only because it's carry...,2
2,2024-07-05 15:19:00,He stated that most of the incidents recorded ...,0
3,2023-11-03 06:37:34,The critical discussions would remain on struc...,2
4,2023-11-02 11:39:18,Topics related to the future of CXOs explore t...,1


# 12

In [13]:
article_clusters = []

for date, group in test_df.groupby("date"):

    counts = group["cluster"].value_counts(normalize=True)

    dist = np.zeros(NUM_CLUSTERS)

    for c,v in counts.items():
        dist[c] = v

    article_clusters.append({
        "date": date,
        "distribution": dist,
        "sentences": group["main_sentence"].tolist()
    })

article_df = pd.DataFrame(article_clusters)

article_df = article_df.sort_values("date")

article_df

,date,distribution,sentences
0,2015-04-21 19:58:14,"[1.0, 0.0, 0.0, 0.0, 0.0]",[Hide Caption 5 of 11 Photos: Effects of globa...
1,2015-09-16 17:26:18,"[0.0, 0.0, 1.0, 0.0, 0.0]",[Beijingers coined the wry phrase '#APEC blue'...
2,2019-02-12 16:03:41,"[0.0, 0.0, 1.0, 0.0, 0.0]","[""Walsh said the authors are right to point ou..."
3,2019-03-26 09:49:53,"[1.0, 0.0, 0.0, 0.0, 0.0]",[He says it is vital to reduce manmad-e greenh...
4,2019-05-08 00:41:43,"[0.0, 0.0, 1.0, 0.0, 0.0]",[But critics say that's only because it's carr...
5,2019-06-09 04:51:35,"[0.0, 0.0, 0.0, 1.0, 0.0]","[""I want people to learn from the challenges w..."
6,2022-03-07 22:30:28,"[1.0, 0.0, 0.0, 0.0, 0.0]",[Inaction or complacency in the setting of an ...
7,2023-11-02 09:06:02,"[0.0, 1.0, 0.0, 0.0, 0.0]",[the unique process solves two critical issues...
8,2023-11-02 11:39:18,"[0.0, 1.0, 0.0, 0.0, 0.0]",[Topics related to the future of CXOs explore ...
9,2023-11-03 06:37:34,"[0.0, 0.0, 1.0, 0.0, 0.0]",[The critical discussions would remain on stru...


# 13

In [14]:
print("\n===== Narrative Drift Detection =====")

for i in range(len(article_df)-1):

    p = article_df.iloc[i]["distribution"]
    q = article_df.iloc[i+1]["distribution"]

    drift = jensenshannon(p,q)

    if drift > 0.3:

        print("\nNarrative Shift Detected")

        print("From:", article_df.iloc[i]["date"])
        print("To:", article_df.iloc[i+1]["date"])

        print("Drift Score:", drift)

        print("\nBefore Narrative:")
        for s in article_df.iloc[i]["sentences"]:
            print("-",s)

        print("\nAfter Narrative:")
        for s in article_df.iloc[i+1]["sentences"]:
            print("-",s)


===== Narrative Drift Detection =====

Narrative Shift Detected
From: 2015-04-21 19:58:14
To: 2015-09-16 17:26:18
Drift Score: 0.8325546111576977

Before Narrative:
- Hide Caption 5 of 11 Photos: Effects of global warming around the worldPollen allergies – Are you sneezing more often these days?

After Narrative:
- Beijingers coined the wry phrase '#APEC blue' after authorities used extreme measures to control pollution during November's APEC summit.

Narrative Shift Detected
From: 2019-02-12 16:03:41
To: 2019-03-26 09:49:53
Drift Score: 0.8325546111576977

Before Narrative:
- "Walsh said the authors are right to point out that the climate-analog approach needs further testing to see if this form of communication works, but, Walsh said, "to a rhetorician of climate, at least, who cares most about promoting democratic deliberation and policymaking around the issue, (the researchers') analogical approach is really promising.

After Narrative:
- He says it is vital to reduce manmad-e gre